In [9]:
import os, json, copy, math, random, warnings, hashlib, gc, textwrap, re
from datetime import datetime, timezone
from pathlib import Path
from typing import Dict, List, Optional, Tuple

from joblib import dump, load
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from IPython.display import display

from sklearn.decomposition import PCA as CPU_PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor

from xgboost import XGBRegressor
import shap
from captum.attr import IntegratedGradients
from gprofiler import GProfiler
import gseapy as gp
from torch_geometric.data import Data
from torch_geometric.nn import GATConv, global_mean_pool
from torch_geometric.explain import Explainer
from torch_geometric.explain.algorithm import GNNExplainer, CaptumExplainer

HAS_XGB = True
HAS_SHAP = True
HAS_CAPTUM = True
HAS_GPROFILER = True
HAS_GSEAPY = True
HAS_PYG = True

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE_SEED = 19537
ALL_SEEDS = [19537, 1584678, 17052356]
SEED = BASE_SEED
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
RNG = np.random.default_rng(SEED)

def set_seeds(seed: int) -> None:
    global SEED, RNG
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    RNG = np.random.default_rng(SEED)
    os.environ["PYTHONHASHSEED"] = str(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Torch device:", TORCH_DEVICE)
print("HAS_XGB:", HAS_XGB, "HAS_SHAP:", HAS_SHAP, "HAS_CAPTUM:", HAS_CAPTUM, "HAS_PYG:", HAS_PYG)

Torch device: cuda
HAS_XGB: True HAS_SHAP: True HAS_CAPTUM: True HAS_PYG: True


In [10]:

# Paths and configs
ARTIFACTS = Path("artifacts")
IN_CLEAN = ARTIFACTS / "cleaned" / "notebook 2"
IN_T2 = ARTIFACTS / "aligned" / "notebook 1" / "track2_nonintersection"
IN_META = ARTIFACTS / "metadata"

NOTEBOOK_SUBDIR = "notebook 9_missingness_threshold_sensitivity"
OUT_REPORTS = ARTIFACTS / "reports" / NOTEBOOK_SUBDIR
OUT_META = ARTIFACTS / "metadata" / NOTEBOOK_SUBDIR
OUT_CACHE = ARTIFACTS / "cache" / NOTEBOOK_SUBDIR
FIG_DIR = OUT_REPORTS / "figures"
INTERP_DIR = OUT_REPORTS / "interpretability"
for d in [OUT_REPORTS, OUT_META, OUT_CACHE, FIG_DIR, INTERP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

LOCK_PATH = IN_META / "proteomics_backbone_lock.json"
if not LOCK_PATH.exists():
    raise FileNotFoundError(f"Missing {LOCK_PATH}")
with LOCK_PATH.open("r", encoding="utf-8") as f:
    backbone_lock = json.load(f)

PRIMARY_ARM = backbone_lock["track1_primary_arm"]
SECONDARY_ARM = backbone_lock["track1_secondary_arm"]
TRACK2_ARM = backbone_lock["track2_stress_test_arm"]
DEPRIORITISED_ARM = backbone_lock.get("deprioritised_arm", "prot_rppa_ccle")

ACTIVE_ARMS = [
    "prot_combined_union",
    "prot_procan_depmapSanger",
    "prot_rppa_ccle",
    "prot_ms_ccle_gygi",
]
PROT_FEATURE_SETS = [
    "prot",
    "rna+prot",
    "cnv+prot",
    "mut+prot",
    "rna+cnv+prot",
    "rna+mut+prot",
    "cnv+mut+prot",
    "rna+cnv+mut+prot",
]
ACTIVE_MODELS = ["ridge", "elasticnet", "extratrees", "xgb", "mlp"]
MISSINGNESS_THRESHOLDS = [0.50, 0.60, 0.70, 0.80, 0.85, 0.90, 0.95]

PRIMARY_TARGET = "auc"
N_DRUGS_BAKEOFF = 100
MIN_CELLS_PER_DRUG = 80
N_SPLITS_DESIRED = 10
USE_PCA = True
PCA_COMPONENTS = 100

RIDGE_ALPHA = 1.0
EN_ALPHA = 0.05
EN_L1_RATIO = 0.2
ET_N_ESTIMATORS = 400
ET_MAX_DEPTH = None
ET_MIN_SAMPLES_LEAF = 2

USE_GPU_FOR_XGB = torch.cuda.is_available()
XGB_DEVICE = "cuda" if USE_GPU_FOR_XGB else "cpu"
XGB_PARAMS = dict(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=SEED, tree_method="hist", device=XGB_DEVICE, n_jobs=-1,
)

MLP_HIDDEN_DIMS = [256, 64]
MLP_DROPOUT = 0.15
MLP_LR = 1e-3
MLP_WEIGHT_DECAY = 1e-5
MLP_EPOCHS = 120
MLP_PATIENCE = 15
MLP_BATCH_SIZE = 64
VAL_FRAC = 0.15
MIN_VAL_SAMPLES = 24

PRINT_EVERY_N_DRUGS = 5
CHECKPOINT_EVERY_N_DRUGS = 5
THRESHOLD_LOCK_FILENAME = "missingness_threshold_choice_prot.json"

MISSINGNESS_TEST_CONFIGS = [
    {"rank": rank, "arm": arm, "model": model, "feature_set": fs}
    for rank, (arm, model, fs) in enumerate(
        [(arm, model, fs) for arm in ACTIVE_ARMS for model in ACTIVE_MODELS for fs in PROT_FEATURE_SETS],
        start=1,
    )
]
CONFIGS_BY_ARM = {arm: [cfg for cfg in MISSINGNESS_TEST_CONFIGS if cfg["arm"] == arm] for arm in ACTIVE_ARMS}
EXPECTED_CONFIG_KEYS = {
    (int(cfg["rank"]), str(cfg["arm"]), str(cfg["model"]).lower(), str(cfg["feature_set"]))
    for cfg in MISSINGNESS_TEST_CONFIGS
}
EXPECTED_CONFIG_KEYS_BY_ARM = {
    arm: {(int(cfg["rank"]), str(cfg["arm"]), str(cfg["model"]).lower(), str(cfg["feature_set"])) for cfg in cfgs}
    for arm, cfgs in CONFIGS_BY_ARM.items()
}
display(pd.DataFrame(MISSINGNESS_TEST_CONFIGS).head(12))

,rank,arm,model,feature_set
0,1,prot_combined_union,ridge,prot
1,2,prot_combined_union,ridge,rna+prot
2,3,prot_combined_union,ridge,cnv+prot
3,4,prot_combined_union,ridge,mut+prot
4,5,prot_combined_union,ridge,rna+cnv+prot
5,6,prot_combined_union,ridge,rna+mut+prot
6,7,prot_combined_union,ridge,cnv+mut+prot
7,8,prot_combined_union,ridge,rna+cnv+mut+prot
8,9,prot_combined_union,elasticnet,prot
9,10,prot_combined_union,elasticnet,rna+prot


In [11]:

# Helpers

def read_parquet_strict(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing parquet: {path}")
    return pd.read_parquet(path)

def normalise_str_index(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.index = df.index.astype(str)
    return df

def write_json(obj: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)

def safe_nanmean(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float); x = x[np.isfinite(x)]
    return float(x.mean()) if x.size else np.nan

def safe_nanmedian(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float); x = x[np.isfinite(x)]
    return float(np.median(x)) if x.size else np.nan

def safe_nanstd(x) -> float:
    x = pd.Series(x).to_numpy(dtype=float); x = x[np.isfinite(x)]
    return float(x.std()) if x.size else np.nan

def spearman_corr(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    if y_true.size < 2:
        return np.nan
    rt = pd.Series(y_true).rank(method="average").to_numpy(dtype=float)
    rp = pd.Series(y_pred).rank(method="average").to_numpy(dtype=float)
    if np.std(rt) == 0 or np.std(rp) == 0:
        return np.nan
    return float(np.corrcoef(rt, rp)[0, 1])

def pick_group_column(cell_index: pd.DataFrame) -> str:
    for c in ["lineage_1", "primary_disease", "lineage", "lineage_2"]:
        if c in cell_index.columns:
            return c
    return "depmap_id"

def safe_group_splits(cells, groups, n_splits_desired, seed):
    groups = groups.reindex(cells).fillna("Unknown").astype(str)
    n_groups, n_cells = groups.nunique(), len(cells)
    n_splits = min(n_splits_desired, n_groups, n_cells)
    if n_splits >= 2 and n_groups >= 2:
        splitter = GroupKFold(n_splits=n_splits)
        return list(splitter.split(np.zeros((n_cells, 1)), np.zeros(n_cells), groups.values)), f"GroupKFold(n_splits={n_splits})"
    n_splits = min(max(2, n_splits), n_cells)
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    return list(splitter.split(np.zeros((n_cells, 1)))), f"KFold(n_splits={n_splits}, shuffle=True, random_state={seed})"

def parse_feature_set(feature_set: str) -> Tuple[str, ...]:
    if feature_set is None:
        return tuple()
    feature_set = str(feature_set).strip()
    return tuple(feature_set.split("+")) if feature_set else tuple()

def concat_selected_modalities(mats: Dict[str, np.ndarray], selected_keys: Tuple[str, ...], n_rows: int) -> np.ndarray:
    parts = []
    for k in selected_keys:
        arr = mats.get(k)
        if arr is None or arr.shape[1] == 0:
            return np.zeros((n_rows, 0), dtype=np.float32)
        parts.append(arr)
    return np.concatenate(parts, axis=1).astype(np.float32) if parts else np.zeros((n_rows, 0), dtype=np.float32)

def wrap_label(s: str, width: int = 24) -> str:
    return "\n".join(textwrap.wrap(str(s), width=width))

def finish_plot(path: Path) -> None:
    plt.tight_layout()
    plt.savefig(path, dpi=170, bbox_inches="tight")
    plt.show()
    plt.close()

def safe_filename(s: str) -> str:
    s = str(s)
    return s.replace(os.sep, "_").replace(" ", "_").replace("+", "plus").replace(":", "_").replace("/", "_").replace("\\", "_").replace("|", "_")

def checkpoint_matches_expected_grid(df: pd.DataFrame, expected_keys) -> bool:
    needed = {"config_rank", "arm", "model", "feature_set"}
    if not needed.issubset(df.columns):
        return False
    found = {
        (int(r["config_rank"]), str(r["arm"]), str(r["model"]).lower(), str(r["feature_set"]))
        for _, r in df[["config_rank", "arm", "model", "feature_set"]].drop_duplicates().iterrows()
    }
    return found.issubset(expected_keys)

def _cache_digest(**parts) -> str:
    payload = json.dumps(parts, sort_keys=True, default=str, separators=(",", ":"))
    return hashlib.sha1(payload.encode("utf-8")).hexdigest()[:20]

def _cache_file(dirpath: Path, prefix: str, **parts) -> Path:
    dirpath.mkdir(parents=True, exist_ok=True)
    digest = _cache_digest(**parts)
    return dirpath / f"{prefix}__{digest}.joblib"

def _eval_cache_dir(kind: str) -> Path:
    p = OUT_CACHE / kind
    p.mkdir(parents=True, exist_ok=True)
    return p

In [12]:

# Model wrappers and transformers

def try_make_xgb(seed: int):
    if not HAS_XGB:
        return None
    try:
        params = dict(XGB_PARAMS); params["random_state"] = seed
        return XGBRegressor(**params)
    except TypeError:
        try:
            params = dict(XGB_PARAMS); params["random_state"] = seed
            params.pop("device", None)
            params["tree_method"] = "gpu_hist" if USE_GPU_FOR_XGB else "hist"
            if USE_GPU_FOR_XGB:
                params["predictor"] = "gpu_predictor"
            return XGBRegressor(**params)
        except Exception:
            return None
    except Exception:
        return None

class TabularMLPRegressor(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: List[int], dropout: float):
        super().__init__()
        layers, dims = [], [input_dim] + list(hidden_dims)
        for din, dout in zip(dims[:-1], dims[1:]):
            layers += [nn.Linear(din, dout), nn.ReLU(), nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-1], 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)

class MLPRegressorWrapper:
    def __init__(self, input_dim, hidden_dims, dropout, lr, weight_decay, epochs, patience, batch_size, seed, device):
        self.input_dim = int(input_dim); self.hidden_dims = list(hidden_dims); self.dropout = float(dropout)
        self.lr = float(lr); self.weight_decay = float(weight_decay); self.epochs = int(epochs)
        self.patience = int(patience); self.batch_size = int(batch_size); self.seed = int(seed); self.device = device
        self.model = None
    def _split_train_val_idx(self, n):
        idx = np.arange(n)
        if n < (MIN_VAL_SAMPLES * 2):
            return idx, np.array([], dtype=int)
        rng = np.random.default_rng(self.seed); rng.shuffle(idx)
        n_val = max(MIN_VAL_SAMPLES, int(round(n * VAL_FRAC))); n_val = min(n_val, n // 3)
        return np.sort(idx[n_val:]), np.sort(idx[:n_val])
    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32); y = np.asarray(y, dtype=np.float32)
        set_seeds(self.seed)
        self.model = TabularMLPRegressor(X.shape[1], self.hidden_dims, self.dropout).to(self.device)
        tr_idx, val_idx = self._split_train_val_idx(X.shape[0])
        ds = TensorDataset(torch.from_numpy(X[tr_idx]), torch.from_numpy(y[tr_idx]))
        dl = DataLoader(ds, batch_size=min(self.batch_size, len(ds)), shuffle=True)
        x_val = torch.from_numpy(X[val_idx]).to(self.device) if len(val_idx) > 0 else None
        y_val = torch.from_numpy(y[val_idx]).to(self.device) if len(val_idx) > 0 else None
        opt = torch.optim.Adam(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()
        best_state = copy.deepcopy(self.model.state_dict()); best_loss = math.inf; bad_epochs = 0
        for _ in range(self.epochs):
            self.model.train()
            for xb, yb in dl:
                xb = xb.to(self.device); yb = yb.to(self.device)
                loss = loss_fn(self.model(xb), yb)
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(self.model.parameters(), 5.0); opt.step()
            self.model.eval()
            with torch.no_grad():
                current = float(loss_fn(self.model(x_val), y_val).item()) if x_val is not None else float(loss.item())
            if current < (best_loss - 1e-6):
                best_loss = current; best_state = copy.deepcopy(self.model.state_dict()); bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= self.patience:
                    break
        self.model.load_state_dict(best_state); self.model.eval(); return self
    def predict(self, X):
        with torch.no_grad():
            return self.model(torch.from_numpy(np.asarray(X, dtype=np.float32)).to(self.device)).detach().cpu().numpy().astype(np.float32)

def make_model(model_name: str, seed: int, input_dim: int):
    model_name = str(model_name).lower()
    if model_name == "ridge":
        return Ridge(alpha=RIDGE_ALPHA, solver="svd")
    if model_name == "elasticnet":
        return ElasticNet(alpha=EN_ALPHA, l1_ratio=EN_L1_RATIO, random_state=seed, max_iter=10000)
    if model_name == "extratrees":
        return ExtraTreesRegressor(n_estimators=ET_N_ESTIMATORS, random_state=seed, n_jobs=-1, max_depth=ET_MAX_DEPTH, min_samples_leaf=ET_MIN_SAMPLES_LEAF)
    if model_name == "xgb":
        return try_make_xgb(seed)
    if model_name == "mlp":
        return MLPRegressorWrapper(input_dim, MLP_HIDDEN_DIMS, MLP_DROPOUT, MLP_LR, MLP_WEIGHT_DECAY, MLP_EPOCHS, MLP_PATIENCE, MLP_BATCH_SIZE, seed, TORCH_DEVICE)
    raise ValueError(model_name)

class BaseBlockTransformer:
    def __init__(self, use_pca: bool, n_components: int, random_state: int):
        self.use_pca = bool(use_pca); self.n_components = int(n_components); self.random_state = int(random_state)
        self.imputer = SimpleImputer(strategy="median"); self.scaler = StandardScaler(with_mean=True, with_std=True); self.pca = None; self.keep_mask = None
    def fit(self, X_train: np.ndarray):
        X_train = np.asarray(X_train, dtype=float)
        self.keep_mask = np.isfinite(X_train).any(axis=0)
        if int(self.keep_mask.sum()) == 0:
            return self
        X_imp = self.imputer.fit_transform(X_train[:, self.keep_mask]); X_std = self.scaler.fit_transform(X_imp)
        if self.use_pca:
            n, d = X_std.shape; n_comp = min(self.n_components, max(1, n - 1), d)
            self.pca = CPU_PCA(n_components=n_comp, random_state=self.random_state); self.pca.fit(X_std)
        return self
    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if self.keep_mask is None or int(self.keep_mask.sum()) == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)
        X_std = self.scaler.transform(self.imputer.transform(X[:, self.keep_mask]))
        if self.pca is not None:
            X_std = self.pca.transform(X_std)
        return X_std.astype(np.float32)

class ThresholdedProtTransformer:
    def __init__(self, max_missing_frac, use_pca, n_components, random_state, add_indicators=False, force_indicators=False):
        self.max_missing_frac = float(max_missing_frac); self.use_pca = bool(use_pca); self.n_components = int(n_components)
        self.random_state = int(random_state); self.add_indicators = bool(add_indicators); self.force_indicators = bool(force_indicators)
        self.imputer = SimpleImputer(strategy="median"); self.scaler = StandardScaler(with_mean=True, with_std=True)
        self.indicator_scaler = None; self.indicator_cols_mask = None; self.pca = None; self.keep_mask = None
        self.n_features_before = 0; self.n_features_kept = 0; self.n_indicator_cols = 0; self.n_output_dims = 0
    def fit(self, X_train: np.ndarray):
        X_train = np.asarray(X_train, dtype=float); self.n_features_before = int(X_train.shape[1])
        miss_frac = np.mean(~np.isfinite(X_train), axis=0); keep_any = np.isfinite(X_train).any(axis=0)
        self.keep_mask = (keep_any & (miss_frac <= self.max_missing_frac))
        self.n_features_kept = int(self.keep_mask.sum())
        if self.n_features_kept == 0:
            return self
        Xk = X_train[:, self.keep_mask]; miss_k = ~np.isfinite(Xk)
        X_std = self.scaler.fit_transform(self.imputer.fit_transform(Xk))
        if self.add_indicators or self.force_indicators:
            self.indicator_cols_mask = miss_k.any(axis=0)
            if int(self.indicator_cols_mask.sum()) > 0:
                ind_tr = miss_k[:, self.indicator_cols_mask].astype(np.float32)
                self.indicator_scaler = StandardScaler(); ind_tr = self.indicator_scaler.fit_transform(ind_tr).astype(np.float32)
                X_std = np.concatenate([X_std, ind_tr], axis=1); self.n_indicator_cols = int(ind_tr.shape[1])
        if self.use_pca:
            n, d = X_std.shape; n_comp = min(self.n_components, max(1, n - 1), d)
            self.pca = CPU_PCA(n_components=n_comp, random_state=self.random_state); self.pca.fit(X_std); self.n_output_dims = int(n_comp)
        else:
            self.n_output_dims = int(X_std.shape[1])
        return self
    def transform(self, X: np.ndarray) -> np.ndarray:
        X = np.asarray(X, dtype=float)
        if self.keep_mask is None or int(self.keep_mask.sum()) == 0:
            return np.zeros((X.shape[0], 0), dtype=np.float32)
        Xk = X[:, self.keep_mask]; miss_k = ~np.isfinite(Xk)
        X_std = self.scaler.transform(self.imputer.transform(Xk))
        if (self.add_indicators or self.force_indicators) and self.indicator_cols_mask is not None and int(self.indicator_cols_mask.sum()) > 0:
            ind = miss_k[:, self.indicator_cols_mask].astype(np.float32); ind = self.indicator_scaler.transform(ind).astype(np.float32)
            X_std = np.concatenate([X_std, ind], axis=1)
        if self.pca is not None:
            X_std = self.pca.transform(X_std)
        return X_std.astype(np.float32)
    def info(self):
        return {"max_missing_frac": self.max_missing_frac, "n_features_before": int(self.n_features_before), "n_features_kept": int(self.n_features_kept), "n_indicator_cols": int(self.n_indicator_cols), "n_output_dims": int(self.n_output_dims), "add_indicators": bool(self.add_indicators), "force_indicators": bool(self.force_indicators)}

In [13]:

# Cache helpers + data load

CACHE_VERSION = "v1_missingness_threshold"
CACHE_TAG = (
    f"{CACHE_VERSION}__target{PRIMARY_TARGET}__splits{N_SPLITS_DESIRED}__ndrugs{N_DRUGS_BAKEOFF}"
    f"__mindrug{MIN_CELLS_PER_DRUG}__pca{int(USE_PCA)}_{PCA_COMPONENTS}"
    f"__ridge{RIDGE_ALPHA}__ena{EN_ALPHA}__enl1{EN_L1_RATIO}__etn{ET_N_ESTIMATORS}"
    f"__mlph{'-'.join(map(str, MLP_HIDDEN_DIMS))}"
)

def _base_cache_path(seed, arm, fold_i): return _cache_file(OUT_CACHE, "base", cache_tag=CACHE_TAG, seed=seed, arm=arm, fold_i=fold_i)
def _prot_threshold_cache_path(seed, arm, fold_i, threshold, add_indicators, force_indicators):
    return _cache_file(OUT_CACHE, "prot_threshold", cache_tag=CACHE_TAG, seed=seed, arm=arm, fold_i=fold_i, threshold=round(float(threshold), 4), add_indicators=bool(add_indicators), force_indicators=bool(force_indicators))
def _eval_cache_path(kind, seed, arm, drug, fold_i, cfg_rank, model_name, feature_set, prot_max_missing_frac, imputer_strategy):
    return _cache_file(_eval_cache_dir(kind), "eval", cache_tag=CACHE_TAG, kind=kind, seed=seed, arm=arm, drug=drug, fold_i=fold_i, cfg_rank=cfg_rank, model_name=model_name, feature_set=feature_set, prot_max_missing_frac=round(float(prot_max_missing_frac),4), imputer_strategy=imputer_strategy)

def load_or_build_base_fold_cache(seed, arm, fold_i, train_cells, eligible_cells):
    path = _base_cache_path(seed, arm, fold_i)
    if path.exists(): return load(path)
    payload = {
        "Xr": BaseBlockTransformer(USE_PCA, PCA_COMPONENTS, seed + 0).fit(rna.loc[train_cells].to_numpy(dtype=float)).transform(rna.loc[eligible_cells].to_numpy(dtype=float)),
        "Xc": BaseBlockTransformer(USE_PCA, PCA_COMPONENTS, seed + 1).fit(cnv.loc[train_cells].to_numpy(dtype=float)).transform(cnv.loc[eligible_cells].to_numpy(dtype=float)),
        "Xm": BaseBlockTransformer(USE_PCA, PCA_COMPONENTS, seed + 2).fit(mut.loc[train_cells].to_numpy(dtype=float)).transform(mut.loc[eligible_cells].to_numpy(dtype=float)),
    }
    dump(payload, path); return payload

def load_or_build_prot_threshold_fold_cache(seed, arm, fold_i, threshold, add_indicators, force_indicators, prot_elig_values, idx_train_full):
    path = _prot_threshold_cache_path(seed, arm, fold_i, threshold, add_indicators, force_indicators)
    if path.exists(): return load(path)
    tr = ThresholdedProtTransformer(threshold, USE_PCA, PCA_COMPONENTS, seed + 300, add_indicators, force_indicators).fit(prot_elig_values[idx_train_full])
    payload = {"Xp": tr.transform(prot_elig_values), "info": tr.info()}
    dump(payload, path); return payload

def load_or_run_eval_cache(*, kind, seed, arm, drug, fold_i, cfg_rank, model_name, feature_set, prot_max_missing_frac, imputer_strategy, extra_meta, X_train, X_test, y_train, y_test):
    path = _eval_cache_path(kind, seed, arm, drug, fold_i, cfg_rank, model_name, feature_set, prot_max_missing_frac, imputer_strategy)
    if path.exists(): return load(path)
    mdl = make_model(model_name, seed, X_train.shape[1])
    if mdl is None:
        row = {"seed": seed, "config_rank": cfg_rank, "arm": arm, "model": model_name, "feature_set": feature_set, "compound_id": drug, "fold": fold_i, "prot_max_missing_frac": float(prot_max_missing_frac), "imputer_strategy": imputer_strategy, "n_train": int(len(y_train)), "n_test": int(len(y_test)), "spearman": np.nan, "r2": np.nan, "fit_status": "model_unavailable", **extra_meta}
        dump(row, path); return row
    mdl.fit(X_train, y_train); pred = mdl.predict(X_test)
    row = {"seed": seed, "config_rank": cfg_rank, "arm": arm, "model": model_name, "feature_set": feature_set, "compound_id": drug, "fold": fold_i, "prot_max_missing_frac": float(prot_max_missing_frac), "imputer_strategy": imputer_strategy, "n_train": int(len(y_train)), "n_test": int(len(y_test)), "spearman": spearman_corr(y_test, pred), "r2": float(r2_score(y_test, pred)), "fit_status": "ok", **extra_meta}
    dump(row, path); return row

cell_index = read_parquet_strict(IN_CLEAN / "cell_index.parquet")
prism_long = read_parquet_strict(IN_CLEAN / "prism_long.parquet")
rna = normalise_str_index(read_parquet_strict(IN_T2 / "rna.parquet"))
cnv = normalise_str_index(read_parquet_strict(IN_T2 / "cnv.parquet"))
mut = normalise_str_index(read_parquet_strict(IN_T2 / "mut.parquet"))
cell_index["depmap_id"] = cell_index["depmap_id"].astype(str).str.strip()
group_col = pick_group_column(cell_index)
groups_all = cell_index.set_index("depmap_id")[group_col].astype("string").fillna("Unknown").astype(str)
core_cells = sorted(set(rna.index) & set(cnv.index) & set(mut.index))
prism_long["depmap_id"] = prism_long["depmap_id"].astype(str).str.strip()
prism_long["compound_id"] = prism_long["compound_id"].astype(str).str.strip()
prism_long["target"] = prism_long["target"].astype(str).str.strip().str.lower()
prism_auc = prism_long[prism_long["target"] == PRIMARY_TARGET][["depmap_id", "compound_id", "y"]].copy()
prot_arm_data = {}
for arm in ACTIVE_ARMS:
    p = IN_T2 / f"prot_optional__{arm}.parquet"
    if p.exists():
        prot_arm_data[arm] = normalise_str_index(pd.read_parquet(p))
        print(f"Loaded {arm}: {prot_arm_data[arm].shape}")
    else:
        print(f"[WARN] {arm} missing at {p}")
drug_cov = prism_auc.groupby("compound_id")["depmap_id"].nunique().sort_values(ascending=False)
selected_drugs = drug_cov.head(N_DRUGS_BAKEOFF).index.tolist()
prism_sel = prism_auc[prism_auc["compound_id"].isin(selected_drugs)].copy()
drug_to_pairs = {k: v for k, v in prism_sel.groupby("compound_id", sort=False)}
print("Core cohort:", len(core_cells), "Selected drugs:", len(selected_drugs))

Loaded prot_combined_union: (1079, 18751)
Loaded prot_procan_depmapSanger: (1079, 7906)
Loaded prot_rppa_ccle: (1079, 144)
Loaded prot_ms_ccle_gygi: (1079, 11780)
Core cohort: 1079 Selected drugs: 100


In [14]:

# Missingness diagnostics
avail_rows = {"rna": {}, "cnv": {}, "mut": {}}
for c in core_cells:
    avail_rows["rna"][c] = 1 if c in rna.index else 0
    avail_rows["cnv"][c] = 1 if c in cnv.index else 0
    avail_rows["mut"][c] = 1 if c in mut.index else 0
for arm, df in prot_arm_data.items():
    avail_rows[arm] = df.reindex(core_cells).notna().any(axis=1).astype(int).to_dict()
availability = pd.DataFrame(avail_rows, index=core_cells); availability.index.name = "depmap_id"
availability.to_csv(OUT_REPORTS / "modality_availability_matrix.csv")
avail_summary = availability.sum().rename("n_cells_present").to_frame().assign(n_cells_total=len(core_cells), pct_present=lambda d: (d["n_cells_present"] / len(core_cells) * 100).round(2))
avail_summary.to_csv(OUT_REPORTS / "modality_availability_summary.csv")
display(avail_summary)

feat_miss_rows, threshold_retention_rows = [], []
for arm, df in prot_arm_data.items():
    df_core = df.reindex(core_cells)
    col_miss, row_miss = df_core.isna().mean(), df_core.isna().mean(axis=1)
    feat_miss_rows.append({"arm": arm, "n_features": int(df_core.shape[1]), "n_cells_in_core": int(df_core.shape[0]), "overall_missing_pct": float(df_core.isna().mean().mean() * 100.0), "col_miss_q10": float(col_miss.quantile(0.10)), "col_miss_q25": float(col_miss.quantile(0.25)), "col_miss_q50": float(col_miss.quantile(0.50)), "col_miss_q75": float(col_miss.quantile(0.75)), "col_miss_q90": float(col_miss.quantile(0.90)), "col_miss_q99": float(col_miss.quantile(0.99)), "row_miss_q10": float(row_miss.quantile(0.10)), "row_miss_q50": float(row_miss.quantile(0.50)), "row_miss_q90": float(row_miss.quantile(0.90)), "n_cols_fully_missing": int((col_miss == 1.0).sum()), "n_rows_fully_missing": int((row_miss == 1.0).sum()), "n_cols_zero_missing": int((col_miss == 0.0).sum())})
    for thr in MISSINGNESS_THRESHOLDS:
        n_kept = int((col_miss <= thr).sum())
        threshold_retention_rows.append({"arm": arm, "prot_max_missing_frac": float(thr), "n_features_total": int(df_core.shape[1]), "n_features_kept_fullcohort": int(n_kept), "frac_features_kept_fullcohort": float(n_kept / max(1, df_core.shape[1]))})
feat_miss_df = pd.DataFrame(feat_miss_rows).sort_values("arm")
threshold_retention_df = pd.DataFrame(threshold_retention_rows).sort_values(["arm", "prot_max_missing_frac"])
feat_miss_df.to_csv(OUT_REPORTS / "per_arm_feature_missingness.csv", index=False)
threshold_retention_df.to_csv(OUT_REPORTS / "threshold_retention_fullcohort.csv", index=False)
display(feat_miss_df)
display(threshold_retention_df.head(16))

,n_cells_present,n_cells_total,pct_present
rna,1079,1079,100.00
cnv,1079,1079,100.00
mut,1079,1079,100.00
prot_combined_union,679,1079,62.93
prot_procan_depmapSanger,485,1079,44.95
prot_rppa_ccle,612,1079,56.72
prot_ms_ccle_gygi,304,1079,28.17


,arm,n_features,n_cells_in_core,overall_missing_pct,col_miss_q10,col_miss_q25,col_miss_q50,col_miss_q75,col_miss_q90,col_miss_q99,row_miss_q10,row_miss_q50,row_miss_q90,n_cols_fully_missing,n_rows_fully_missing,n_cols_zero_missing
0,prot_combined_union,18751,1079,74.118180,0.556070,0.718258,0.718258,0.825765,0.915663,0.961075,0.249363,0.760706,1.0,0,400,0
3,prot_ms_ccle_gygi,11780,1079,78.876396,0.718258,0.718258,0.732159,0.861214,0.949027,0.979611,0.237267,1.000000,1.0,0,775,0
1,prot_procan_depmapSanger,7906,1079,70.717562,0.550510,0.560704,0.668211,0.843373,0.931418,0.975904,0.308981,1.000000,1.0,0,594,0
2,prot_rppa_ccle,144,1079,43.280816,0.432808,0.432808,0.432808,0.432808,0.432808,0.432808,0.000000,0.000000,1.0,0,467,0


,arm,prot_max_missing_frac,n_features_total,n_features_kept_fullcohort,frac_features_kept_fullcohort
0,prot_combined_union,0.50,18751,144,0.007680
1,prot_combined_union,0.60,18751,3172,0.169164
2,prot_combined_union,0.70,18751,4452,0.237427
3,prot_combined_union,0.80,18751,13374,0.713242
4,prot_combined_union,0.85,18751,14784,0.788438
5,prot_combined_union,0.90,18751,16335,0.871154
6,prot_combined_union,0.95,18751,18198,0.970508
21,prot_ms_ccle_gygi,0.50,11780,0,0.000000
22,prot_ms_ccle_gygi,0.60,11780,0,0.000000
23,prot_ms_ccle_gygi,0.70,11780,0,0.000000


In [15]:

# Combined union platform diagnostics + missingness report
pat_counts = platform_miss_df = None
if TRACK2_ARM in prot_arm_data:
    union_df = prot_arm_data[TRACK2_ARM].reindex(core_cells)
    prefixes = {"ms": "ms__", "rppa": "rppa__", "procan": "procan__"}
    plat_pres = pd.DataFrame(index=union_df.index)
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        plat_pres[key] = union_df[cols].notna().any(axis=1).astype(int) if cols else 0
    def pattern_label(row):
        inc = [k for k in ["ms", "rppa", "procan"] if row.get(k, 0) == 1]
        return "+".join(inc) if inc else "none"
    pat_counts = plat_pres.apply(pattern_label, axis=1).rename("pattern").value_counts().rename_axis("pattern").reset_index(name="n_cells")
    pat_counts["frac_cells"] = pat_counts["n_cells"] / float(union_df.shape[0])
    pm_rows = []
    for key, pref in prefixes.items():
        cols = [c for c in union_df.columns if str(c).startswith(pref)]
        if not cols: continue
        n_absent = int(plat_pres[key].eq(0).sum()); miss_from_abs = n_absent * len(cols); miss_total = int(union_df[cols].isna().sum().sum())
        pm_rows.append({"platform": key, "n_features": len(cols), "n_cells_present": int(plat_pres[key].sum()), "n_cells_absent": n_absent, "frac_cells_present": float(plat_pres[key].mean()), "pct_missingness_from_platform_absence": float(miss_from_abs / miss_total * 100.0) if miss_total > 0 else np.nan})
    platform_miss_df = pd.DataFrame(pm_rows)
    pat_counts.to_csv(OUT_REPORTS / "combined_union_platform_patterns.csv", index=False)
    platform_miss_df.to_csv(OUT_REPORTS / "combined_union_platform_missingness_contrib.csv", index=False)

missingness_report = {
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "seeds": ALL_SEEDS,
    "active_arms": ACTIVE_ARMS,
    "deprioritised_arm": DEPRIORITISED_ARM,
    "threshold_grid": [float(x) for x in MISSINGNESS_THRESHOLDS],
    "leakage_note": "Protein feature missingness thresholding, imputation, scaling, indicators, and PCA are fit inside train folds only.",
    "modality_availability": avail_summary.reset_index().to_dict(orient="records"),
    "per_arm_feature_missingness": feat_miss_df.to_dict(orient="records"),
    "threshold_retention_fullcohort": threshold_retention_df.to_dict(orient="records"),
    "combined_union_platform_patterns": None if pat_counts is None else pat_counts.to_dict(orient="records"),
    "combined_union_platform_missingness_contrib": None if platform_miss_df is None else platform_miss_df.to_dict(orient="records"),
}
write_json(missingness_report, OUT_REPORTS / "missingness_report.json")
print("Wrote missingness report.")

Wrote missingness report.


In [ ]:

# Main benchmark loop with caching + resume
print("MISSINGNESS-THRESHOLD SENSITIVITY BAKE-OFF")
all_bakeoff_rows, seeds_to_run = [], []
REQUIRED_COLS = {"config_rank", "arm", "model", "feature_set", "compound_id", "fold", "prot_max_missing_frac", "imputer_strategy", "uses_prot", "n_prot_features_kept"}

for run_seed in ALL_SEEDS:
    seed_path = OUT_REPORTS / f"missingness_threshold_bakeoff_seed{run_seed}.csv"
    if seed_path.exists():
        existing = pd.read_csv(seed_path)
        if (not REQUIRED_COLS.issubset(existing.columns)) or (not checkpoint_matches_expected_grid(existing, EXPECTED_CONFIG_KEYS)):
            print(f"[resume] Seed {run_seed} file exists but is from old schema/grid, rerunning.")
            seeds_to_run.append(run_seed); continue
        existing["seed"] = pd.to_numeric(existing["seed"], errors="coerce").astype("Int64")
        n_drugs_in_file = existing["compound_id"].nunique()
        if n_drugs_in_file >= N_DRUGS_BAKEOFF:
            print(f"[resume] Seed {run_seed} complete ({n_drugs_in_file} drugs), loading.")
            all_bakeoff_rows.extend(existing.to_dict(orient="records"))
        else:
            print(f"[resume] Seed {run_seed} incomplete ({n_drugs_in_file}/{N_DRUGS_BAKEOFF} drugs), will rerun.")
            seeds_to_run.append(run_seed)
    else:
        seeds_to_run.append(run_seed)

if not seeds_to_run:
    print("[resume] All seeds complete, skipping loop.")
else:
    print("[resume] Seeds remaining:", seeds_to_run)

for run_seed in seeds_to_run:
    set_seeds(run_seed)
    print(f"\nSeed {run_seed}")
    for arm in ACTIVE_ARMS:
        if arm not in prot_arm_data:
            print(f"  [SKIP] {arm} not loaded."); continue
        arm_cfgs = CONFIGS_BY_ARM.get(arm, [])
        prot_df = prot_arm_data[arm].copy(); prot_df.index = prot_df.index.astype(str).str.strip(); prot_core = prot_df.reindex(core_cells)
        eligible_cells = sorted(prot_core.notna().any(axis=1)[lambda s: s].index.tolist())
        if len(eligible_cells) < 200:
            print(f"  [SKIP] {arm}: only {len(eligible_cells)} eligible cells."); continue

        arm_ckpt_path = OUT_REPORTS / f"missingness_threshold_bakeoff_seed{run_seed}_{arm}.csv"
        already_done_drugs = set()
        if arm_ckpt_path.exists():
            arm_existing = pd.read_csv(arm_ckpt_path)
            if REQUIRED_COLS.issubset(arm_existing.columns) and checkpoint_matches_expected_grid(arm_existing, EXPECTED_CONFIG_KEYS_BY_ARM[arm]):
                n_drugs_in_arm = arm_existing["compound_id"].nunique()
                if n_drugs_in_arm >= N_DRUGS_BAKEOFF:
                    print(f"  [resume] seed={run_seed} arm={arm} complete ({n_drugs_in_arm} drugs), loading.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records")); continue
                else:
                    print(f"  [resume] seed={run_seed} arm={arm} partial ({n_drugs_in_arm}/{N_DRUGS_BAKEOFF} drugs), resuming.")
                    all_bakeoff_rows.extend(arm_existing.to_dict(orient="records")); already_done_drugs = set(arm_existing["compound_id"].astype(str).unique().tolist())
            else:
                print(f"  [resume] seed={run_seed} arm={arm} checkpoint is old schema/grid, rerunning arm.")

        arm_groups = groups_all.reindex(eligible_cells).fillna("Unknown").astype(str)
        splits, split_name = safe_group_splits(eligible_cells, arm_groups, N_SPLITS_DESIRED, seed=run_seed)
        print(f"  {arm}: {split_name}, eligible_cells={len(eligible_cells)}")
        fold_map = {}
        for fold_i, (_, test_idx) in enumerate(splits):
            for j in test_idx:
                fold_map[eligible_cells[int(j)]] = fold_i

        prot_elig_values = prot_core.loc[eligible_cells].to_numpy(dtype=float)
        fold_cache = {}
        force_indicators = (arm == TRACK2_ARM)
        print(f"    Building/loading fold caches for {arm} ...")
        for fold_i, (train_idx, _) in enumerate(splits):
            train_cells = [eligible_cells[int(j)] for j in train_idx]
            base_payload = load_or_build_base_fold_cache(run_seed, arm, fold_i, train_cells, eligible_cells)
            prot_by_threshold = {}
            for thr in MISSINGNESS_THRESHOLDS:
                prot_by_threshold[float(thr)] = load_or_build_prot_threshold_fold_cache(run_seed, arm, fold_i, thr, False, force_indicators, prot_elig_values, np.asarray(train_idx, dtype=int))
            fold_cache[fold_i] = {"base_mats": {"rna": base_payload["Xr"], "cnv": base_payload["Xc"], "mut": base_payload["Xm"]}, "prot_by_threshold": prot_by_threshold}

        n_run = n_skip = 0
        for drug_i, drug in enumerate(selected_drugs, start=1):
            if drug in already_done_drugs:
                continue
            pairs = drug_to_pairs.get(drug)
            if pairs is None:
                n_skip += 1; continue
            df = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].copy()
            if df["depmap_id"].nunique() < MIN_CELLS_PER_DRUG:
                n_skip += 1; continue
            df = df.groupby("depmap_id", as_index=False)["y"].mean()
            cell_ids = df["depmap_id"].astype(str).tolist(); y_all = df["y"].to_numpy(dtype=float)
            fold_ids = np.array([fold_map.get(c, -1) for c in cell_ids], dtype=int)
            valid = fold_ids >= 0
            cell_ids = [c for c, v in zip(cell_ids, valid) if v]; y_all = y_all[valid]; fold_ids = fold_ids[valid]
            if len(cell_ids) < MIN_CELLS_PER_DRUG:
                n_skip += 1; continue
            n_run += 1
            if (drug_i % PRINT_EVERY_N_DRUGS == 0) or (drug_i == 1) or (drug_i == len(selected_drugs)):
                print(f"    [{arm}] ({drug_i}/{len(selected_drugs)}) drug={drug} n_cells={len(cell_ids)}")
            if (n_run % CHECKPOINT_EVERY_N_DRUGS) == 0:
                arm_rows_partial = [r for r in all_bakeoff_rows if r["seed"] == run_seed and r["arm"] == arm]
                if arm_rows_partial:
                    pd.DataFrame(arm_rows_partial).to_csv(arm_ckpt_path, index=False)
                    done = pd.DataFrame(arm_rows_partial)["compound_id"].nunique()
                    print(f"      [mid-checkpoint] drug {done}/{N_DRUGS_BAKEOFF} | {arm_ckpt_path}")

            c2r = {c: i for i, c in enumerate(eligible_cells)}
            idx_all = np.array([c2r[c] for c in cell_ids], dtype=int)

            for fold_i, _ in enumerate(splits):
                in_test = fold_ids == fold_i
                n_test, n_train = int(in_test.sum()), int((~in_test).sum())
                if n_test < 5 or n_train < 20:
                    continue
                idx_train, idx_test = idx_all[~in_test], idx_all[in_test]
                y_train, y_test = y_all[~in_test], y_all[in_test]
                base_mats = fold_cache[fold_i]["base_mats"]

                for cfg in arm_cfgs:
                    cfg_rank = int(cfg["rank"])
                    cfg_model = str(cfg["model"]).lower()
                    cfg_feature_set = str(cfg["feature_set"])
                    cfg_keys = parse_feature_set(cfg_feature_set)
                    cfg_keys_wo_prot = tuple(k for k in cfg_keys if k != "prot")
                    X_nonprot = concat_selected_modalities(base_mats, cfg_keys_wo_prot, len(eligible_cells))

                    # Fit the non-proteomics reference once, then reuse it across thresholds
                    shared_ref_row = None
                    if len(cfg_keys_wo_prot) > 0 and X_nonprot.shape[1] > 0:
                        shared_ref_row = load_or_run_eval_cache(
                            kind="missingness_eval",
                            seed=run_seed,
                            arm=arm,
                            drug=drug,
                            fold_i=fold_i,
                            cfg_rank=cfg_rank,
                            model_name=cfg_model,
                            feature_set=cfg_feature_set,
                            prot_max_missing_frac=-1.0,
                            imputer_strategy="reference_without_prot_shared",
                            extra_meta={
                                "uses_prot": True,
                                "add_indicators": False,
                                "force_indicators": force_indicators,
                                "n_prot_features_before": np.nan,
                                "n_prot_features_kept": np.nan,
                                "n_prot_indicator_cols": np.nan,
                                "n_prot_pca_dims": np.nan,
                            },
                            X_train=X_nonprot[idx_train],
                            X_test=X_nonprot[idx_test],
                            y_train=y_train,
                            y_test=y_test,
                        )

                    for thr in MISSINGNESS_THRESHOLDS:
                        thr = float(thr)
                        prot_payload = fold_cache[fold_i]["prot_by_threshold"][thr]
                        Xp, prot_info = prot_payload["Xp"], prot_payload["info"]

                        if shared_ref_row is not None:
                            row_ref = dict(shared_ref_row)
                            row_ref["prot_max_missing_frac"] = thr
                            row_ref["imputer_strategy"] = "reference_without_prot"
                            row_ref["n_prot_features_before"] = int(prot_info["n_features_before"])
                            row_ref["n_prot_features_kept"] = int(prot_info["n_features_kept"])
                            row_ref["n_prot_indicator_cols"] = int(prot_info["n_indicator_cols"])
                            row_ref["n_prot_pca_dims"] = int(prot_info["n_output_dims"])
                            all_bakeoff_rows.append(row_ref)

                        if Xp is None or Xp.shape[1] == 0:
                            all_bakeoff_rows.append({
                                "seed": run_seed,
                                "config_rank": cfg_rank,
                                "arm": arm,
                                "model": cfg_model,
                                "feature_set": cfg_feature_set,
                                "uses_prot": True,
                                "compound_id": drug,
                                "fold": fold_i,
                                "prot_max_missing_frac": thr,
                                "imputer_strategy": "thresholded_prot",
                                "add_indicators": False,
                                "force_indicators": force_indicators,
                                "n_train": n_train,
                                "n_test": n_test,
                                "spearman": np.nan,
                                "r2": np.nan,
                                "fit_status": "empty_prot_after_threshold",
                                "n_prot_features_before": int(prot_info["n_features_before"]),
                                "n_prot_features_kept": int(prot_info["n_features_kept"]),
                                "n_prot_indicator_cols": int(prot_info["n_indicator_cols"]),
                                "n_prot_pca_dims": int(prot_info["n_output_dims"]),
                            })
                            continue

                        parts, bad_cfg = [], False
                        for k in cfg_keys:
                            if k == "prot":
                                parts.append(Xp)
                            else:
                                arr = base_mats.get(k)
                                if arr is None or arr.shape[1] == 0:
                                    bad_cfg = True
                                    break
                                parts.append(arr)

                        if bad_cfg or len(parts) == 0:
                            continue

                        Xf = np.concatenate(parts, axis=1).astype(np.float32)
                        row = load_or_run_eval_cache(
                            kind="missingness_eval",
                            seed=run_seed,
                            arm=arm,
                            drug=drug,
                            fold_i=fold_i,
                            cfg_rank=cfg_rank,
                            model_name=cfg_model,
                            feature_set=cfg_feature_set,
                            prot_max_missing_frac=thr,
                            imputer_strategy="thresholded_prot",
                            extra_meta={
                                "uses_prot": True,
                                "add_indicators": False,
                                "force_indicators": force_indicators,
                                "n_prot_features_before": int(prot_info["n_features_before"]),
                                "n_prot_features_kept": int(prot_info["n_features_kept"]),
                                "n_prot_indicator_cols": int(prot_info["n_indicator_cols"]),
                                "n_prot_pca_dims": int(prot_info["n_output_dims"]),
                            },
                            X_train=Xf[idx_train],
                            X_test=Xf[idx_test],
                            y_train=y_train,
                            y_test=y_test,
                        )
                        all_bakeoff_rows.append(row)

                        if Xp is None or Xp.shape[1] == 0:
                            all_bakeoff_rows.append({"seed": run_seed, "config_rank": cfg_rank, "arm": arm, "model": cfg_model, "feature_set": cfg_feature_set, "uses_prot": True, "compound_id": drug, "fold": fold_i, "prot_max_missing_frac": thr, "imputer_strategy": "thresholded_prot", "add_indicators": False, "force_indicators": force_indicators, "n_train": n_train, "n_test": n_test, "spearman": np.nan, "r2": np.nan, "fit_status": "empty_prot_after_threshold", "n_prot_features_before": int(prot_info["n_features_before"]), "n_prot_features_kept": int(prot_info["n_features_kept"]), "n_prot_indicator_cols": int(prot_info["n_indicator_cols"]), "n_prot_pca_dims": int(prot_info["n_output_dims"])})
                            continue

                        parts, bad_cfg = [], False
                        for k in cfg_keys:
                            if k == "prot":
                                parts.append(Xp)
                            else:
                                arr = base_mats.get(k)
                                if arr is None or arr.shape[1] == 0:
                                    bad_cfg = True; break
                                parts.append(arr)
                        if bad_cfg or len(parts) == 0:
                            continue
                        Xf = np.concatenate(parts, axis=1).astype(np.float32)
                        row = load_or_run_eval_cache(
                            kind="missingness_eval", seed=run_seed, arm=arm, drug=drug, fold_i=fold_i, cfg_rank=cfg_rank,
                            model_name=cfg_model, feature_set=cfg_feature_set, prot_max_missing_frac=thr, imputer_strategy="thresholded_prot",
                            extra_meta={"uses_prot": True, "add_indicators": False, "force_indicators": force_indicators, "n_prot_features_before": int(prot_info["n_features_before"]), "n_prot_features_kept": int(prot_info["n_features_kept"]), "n_prot_indicator_cols": int(prot_info["n_indicator_cols"]), "n_prot_pca_dims": int(prot_info["n_output_dims"])},
                            X_train=Xf[idx_train], X_test=Xf[idx_test], y_train=y_train, y_test=y_test
                        )
                        all_bakeoff_rows.append(row)

            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        print(f"    [{arm}] drugs_run={n_run}, drugs_skipped={n_skip}")
        pd.DataFrame([r for r in all_bakeoff_rows if r["seed"] == run_seed and r["arm"] == arm]).to_csv(arm_ckpt_path, index=False)
        print(f"    [checkpoint] {arm_ckpt_path}")

    seed_df = pd.DataFrame([r for r in all_bakeoff_rows if r["seed"] == run_seed])
    seed_path = OUT_REPORTS / f"missingness_threshold_bakeoff_seed{run_seed}.csv"
    seed_df.to_csv(seed_path, index=False)
    print(f"  Saved seed {run_seed}: {seed_path}  shape={seed_df.shape}")

MISSINGNESS-THRESHOLD SENSITIVITY BAKE-OFF
[resume] Seeds remaining: [19537, 1584678, 17052356]

=== Seed 19537 ===
  prot_combined_union: GroupKFold(n_splits=10), eligible_cells=679
    Building/loading fold caches for prot_combined_union ...
    [prot_combined_union] (1/100) drug=IXAZOMIB (BRD:BRD-K78659596-001-03-9) n_cells=371


KeyboardInterrupt: 

In [ ]:

# Summaries and lock decision
bakeoff_df = pd.DataFrame(all_bakeoff_rows)
merged_path = OUT_REPORTS / f"missingness_threshold_bakeoff_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_df.to_csv(merged_path, index=False)
print("Merged:", merged_path, "shape=", bakeoff_df.shape)
if bakeoff_df.empty:
    raise RuntimeError("No rows were produced.")

drug_means = (
    bakeoff_df
    .groupby(["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "prot_max_missing_frac", "imputer_strategy", "compound_id"], as_index=False)
    .agg(
        spearman=("spearman", safe_nanmean),
        r2=("r2", safe_nanmean),
        n_folds=("fold", "nunique"),
        mean_n_prot_features_kept=("n_prot_features_kept", safe_nanmean),
        mean_n_prot_pca_dims=("n_prot_pca_dims", safe_nanmean),
    )
)

bakeoff_summary = (
    drug_means
    .groupby(["config_rank", "arm", "model", "feature_set", "uses_prot", "prot_max_missing_frac", "imputer_strategy"], as_index=False)
    .agg(
        n_seeds=("seed", "nunique"),
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", safe_nanmean),
        median_spearman=("spearman", safe_nanmedian),
        std_spearman=("spearman", safe_nanstd),
        mean_r2=("r2", safe_nanmean),
        mean_n_prot_features_kept=("mean_n_prot_features_kept", safe_nanmean),
        mean_n_prot_pca_dims=("mean_n_prot_pca_dims", safe_nanmean),
    )
)

base_ref = (
    bakeoff_summary[bakeoff_summary["imputer_strategy"] == "reference_without_prot"][["config_rank", "arm", "model", "feature_set", "prot_max_missing_frac", "mean_spearman"]]
    .rename(columns={"mean_spearman": "baseline_mean_spearman"})
    .drop_duplicates(subset=["config_rank", "arm", "model", "feature_set", "prot_max_missing_frac"])
)
bakeoff_summary = bakeoff_summary.merge(base_ref, on=["config_rank", "arm", "model", "feature_set", "prot_max_missing_frac"], how="left")
bakeoff_summary["delta_vs_baseline"] = np.where(bakeoff_summary["imputer_strategy"] == "reference_without_prot", 0.0, bakeoff_summary["mean_spearman"] - bakeoff_summary["baseline_mean_spearman"])

summary_path = OUT_REPORTS / f"missingness_threshold_bakeoff_summary_merged_{len(ALL_SEEDS)}seeds.csv"
bakeoff_summary.to_csv(summary_path, index=False)
display(bakeoff_summary.head(24))

per_seed_summary = (
    drug_means
    .groupby(["seed", "config_rank", "arm", "model", "feature_set", "uses_prot", "prot_max_missing_frac", "imputer_strategy"], as_index=False)
    .agg(
        n_drugs=("compound_id", "nunique"),
        mean_spearman=("spearman", safe_nanmean),
        median_spearman=("spearman", safe_nanmedian),
        std_spearman=("spearman", safe_nanstd),
        mean_r2=("r2", safe_nanmean),
    )
)
per_seed_summary.to_csv(OUT_REPORTS / "missingness_threshold_bakeoff_per_seed_summary.csv", index=False)

prot_only_summary = bakeoff_summary[bakeoff_summary["imputer_strategy"] == "thresholded_prot"].copy()
best_threshold_per_config = (
    prot_only_summary.sort_values(["config_rank", "mean_spearman", "median_spearman", "mean_r2", "prot_max_missing_frac"], ascending=[True, False, False, False, True])
    .groupby(["config_rank", "arm", "model", "feature_set"], as_index=False).head(1).reset_index(drop=True)
)
best_threshold_per_config.to_csv(OUT_REPORTS / "best_threshold_per_config.csv", index=False)

arm_threshold_overall = (
    prot_only_summary.groupby(["arm", "prot_max_missing_frac"], as_index=False)
    .agg(
        n_configs=("config_rank", "nunique"),
        mean_spearman=("mean_spearman", safe_nanmean),
        median_spearman=("mean_spearman", safe_nanmedian),
        mean_r2=("mean_r2", safe_nanmean),
        mean_delta_vs_baseline=("delta_vs_baseline", safe_nanmean),
        mean_n_prot_features_kept=("mean_n_prot_features_kept", safe_nanmean),
    )
    .sort_values(["arm", "mean_spearman", "mean_delta_vs_baseline", "mean_r2"], ascending=[True, False, False, False])
)
best_threshold_per_arm = arm_threshold_overall.groupby("arm", as_index=False).head(1).reset_index(drop=True)
best_threshold_per_arm.to_csv(OUT_REPORTS / "best_threshold_per_arm.csv", index=False)

primary_arm_choice = best_threshold_per_arm[best_threshold_per_arm["arm"] == PRIMARY_ARM].copy()

threshold_choice_doc = {
    "locked_at": datetime.now(timezone.utc).isoformat(),
    "seeds_used": ALL_SEEDS,
    "n_drugs_bakeoff": N_DRUGS_BAKEOFF,
    "primary_target": PRIMARY_TARGET,
    "proteomics_arms": ACTIVE_ARMS,
    "models": ACTIVE_MODELS,
    "feature_sets": PROT_FEATURE_SETS,
    "threshold_grid": [float(x) for x in MISSINGNESS_THRESHOLDS],
    "use_pca": bool(USE_PCA),
    "pca_components": int(PCA_COMPONENTS),
    "note": "Protein feature missingness thresholds are evaluated fold-safely; columns above threshold are dropped on the training fold only before fold-safe imputation, scaling, indicators, and PCA.",
    "best_threshold_per_config": best_threshold_per_config.to_dict(orient="records"),
    "best_threshold_per_arm": best_threshold_per_arm.to_dict(orient="records"),
    "primary_arm_recommendation": primary_arm_choice.iloc[0].to_dict() if primary_arm_choice.shape[0] > 0 else None,
}
threshold_choice_path = OUT_META / THRESHOLD_LOCK_FILENAME
write_json(threshold_choice_doc, threshold_choice_path)
write_json(threshold_choice_doc, IN_META / THRESHOLD_LOCK_FILENAME)
print("Wrote lock:", threshold_choice_path)
display(best_threshold_per_arm)

In [ ]:

# EDA plots
threshold_perf = (
    prot_only_summary
    .groupby(["arm", "model", "prot_max_missing_frac"], as_index=False)
    .agg(
        mean_spearman=("mean_spearman", safe_nanmean),
        mean_delta_vs_baseline=("delta_vs_baseline", safe_nanmean),
        mean_r2=("mean_r2", safe_nanmean),
        mean_n_prot_features_kept=("mean_n_prot_features_kept", safe_nanmean),
    )
)
threshold_perf.to_csv(OUT_REPORTS / "threshold_performance_by_arm_model.csv", index=False)

plt.figure(figsize=(8.5, 5.0))
for arm, sub in threshold_retention_df.groupby("arm", sort=False):
    plt.plot(sub["prot_max_missing_frac"], sub["frac_features_kept_fullcohort"], marker="o", label=str(arm))
plt.title("Fraction of proteins retained vs missingness threshold")
plt.xlabel("Maximum allowed protein missingness fraction")
plt.ylabel("Fraction of proteins retained")
plt.legend()
finish_plot(FIG_DIR / "threshold_retention_curves.png")

for arm in ACTIVE_ARMS:
    sub = threshold_perf[threshold_perf["arm"] == arm].copy()
    if sub.empty: 
        continue
    plt.figure(figsize=(8.6, 5.0))
    for model, g in sub.groupby("model", sort=False):
        plt.plot(g["prot_max_missing_frac"], g["mean_spearman"], marker="o", label=str(model))
    plt.title(f"Mean Spearman vs missingness threshold: {arm}")
    plt.xlabel("Maximum allowed protein missingness fraction")
    plt.ylabel("Mean Spearman")
    plt.legend()
    finish_plot(FIG_DIR / f"threshold_curve__{safe_filename(arm)}.png")

best_threshold_table = best_threshold_per_arm[["arm", "prot_max_missing_frac", "mean_spearman", "mean_delta_vs_baseline", "mean_r2", "mean_n_prot_features_kept"]].sort_values("mean_spearman", ascending=False).reset_index(drop=True)
best_threshold_table.to_csv(OUT_REPORTS / "best_threshold_table.csv", index=False)
display(best_threshold_table)

plt.figure(figsize=(8.0, 4.8))
plt.bar(best_threshold_table["arm"], best_threshold_table["mean_spearman"])
for i, (thr, sp) in enumerate(zip(best_threshold_table["prot_max_missing_frac"], best_threshold_table["mean_spearman"])):
    plt.text(i, sp, f"thr={thr:.2f}\nsp={sp:.3f}", ha="center", va="bottom", fontsize=9)
plt.title("Best threshold per arm")
plt.xlabel("Proteomics arm")
plt.ylabel("Mean Spearman")
plt.xticks(rotation=20, ha="right")
finish_plot(FIG_DIR / "best_threshold_per_arm.png")

In [ ]:

# Interpretability helpers

def build_original_filtered_block(df, train_cells, all_cells, prefix, strategy="median", do_scale=True, max_missing_frac=None, add_indicators=False, force_indicators=False):
    X_train = df.loc[train_cells].to_numpy(dtype=float); X_all = df.loc[all_cells].to_numpy(dtype=float)
    keep = np.isfinite(X_train).any(axis=0)
    if max_missing_frac is not None:
        keep = keep & (np.mean(~np.isfinite(X_train), axis=0) <= float(max_missing_frac))
    cols = df.columns[keep].astype(str).tolist()
    if keep.sum() == 0:
        return np.zeros((len(train_cells), 0), dtype=np.float32), np.zeros((len(all_cells), 0), dtype=np.float32), []
    X_train, X_all = X_train[:, keep], X_all[:, keep]
    imp = SimpleImputer(strategy="most_frequent" if strategy == "most_frequent" else "median")
    X_train_imp, X_all_imp = imp.fit_transform(X_train), imp.transform(X_all)
    if do_scale:
        sc = StandardScaler(); X_train_imp, X_all_imp = sc.fit_transform(X_train_imp), sc.transform(X_all_imp)
    names = [f"{prefix}::{c}" for c in cols]
    if add_indicators or force_indicators:
        miss_tr, miss_all = ~np.isfinite(X_train), ~np.isfinite(X_all); ind_any = miss_tr.any(axis=0)
        if int(ind_any.sum()) > 0:
            ind_sc = StandardScaler()
            ind_tr, ind_all = ind_sc.fit_transform(miss_tr[:, ind_any].astype(np.float32)), ind_sc.transform(miss_all[:, ind_any].astype(np.float32))
            X_train_imp, X_all_imp = np.concatenate([X_train_imp, ind_tr], axis=1), np.concatenate([X_all_imp, ind_all], axis=1)
            names += [f"{prefix}::missing_{c}" for c in np.asarray(cols)[ind_any].tolist()]
    return X_train_imp.astype(np.float32), X_all_imp.astype(np.float32), names

def mean_abs_shap_df(shap_values, feature_names):
    arr = np.abs(np.asarray(shap_values, dtype=float)).mean(axis=0)
    return pd.DataFrame({"feature": feature_names, "mean_abs_shap": arr}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

def feature_to_modality(feat: str) -> str:
    return str(feat).split("::", 1)[0] if "::" in str(feat) else "unknown"

def feature_to_gene(feat: str) -> str:
    feat = str(feat).split("::", 1)[1] if "::" in str(feat) else str(feat)
    if feat.startswith("missing_"): feat = feat[len("missing_"):]
    if "__" in feat: feat = feat.split("__")[-1]
    return re.sub(r"[^A-Za-z0-9\-_]", "", feat)

def try_run_pathway_enrichment(feature_names: List[str], out_dir: Path):
    genes = [feature_to_gene(f) for f in feature_names if isinstance(f, str)]
    genes = [g for g in genes if g]; genes = list(dict.fromkeys(genes))[:300]
    out_dir.mkdir(parents=True, exist_ok=True)
    if not genes:
        print("[pathway] No genes."); return
    with (out_dir / "top_genes.txt").open("w", encoding="utf-8") as f:
        for g in genes: f.write(f"{g}\n")
    if HAS_GPROFILER:
        try:
            enr = GProfiler(return_dataframe=True).profile(organism="hsapiens", query=genes, sources=["GO:BP", "REAC"])
            if enr is not None and enr.shape[0] > 0:
                enr.sort_values("p_value", ascending=True).head(30).to_csv(out_dir / "pathway_enrichment_gprofiler.csv", index=False)
                print("[pathway] g:Profiler written."); return
        except Exception as e:
            print("[pathway] g:Profiler unavailable:", e)
    if HAS_GSEAPY:
        try:
            enr = gp.enrichr(gene_list=genes, gene_sets=["GO_Biological_Process_2023", "Reactome_2022"], organism="Human", outdir=None)
            if hasattr(enr, "results") and enr.results is not None:
                enr.results.to_csv(out_dir / "pathway_enrichment_gseapy.csv", index=False); print("[pathway] gseapy written."); return
        except Exception as e:
            print("[pathway] gseapy unavailable:", e)
    print("[pathway] No enrichment backend.")

In [ ]:

# Case-study selection + SHAP + IG + attention surrogate + optional GNN pilot
tree_case_pool = prot_only_summary[prot_only_summary["model"].isin(["extratrees", "xgb"])].copy()
mlp_case_pool = prot_only_summary[prot_only_summary["model"] == "mlp"].copy()
tree_case = tree_case_pool.sort_values(["mean_spearman", "mean_r2", "n_drugs"], ascending=[False, False, False]).iloc[0].to_dict() if not tree_case_pool.empty else None
mlp_case = mlp_case_pool.sort_values(["mean_spearman", "mean_r2", "n_drugs"], ascending=[False, False, False]).iloc[0].to_dict() if not mlp_case_pool.empty else None

def get_best_detail_row_for_case(detail_df, summary_row, seed):
    if summary_row is None: return None
    sub = detail_df[(detail_df["seed"] == seed) & (detail_df["arm"] == summary_row["arm"]) & (detail_df["model"] == summary_row["model"]) & (detail_df["feature_set"] == summary_row["feature_set"]) & (detail_df["prot_max_missing_frac"] == summary_row["prot_max_missing_frac"]) & (detail_df["imputer_strategy"] == "thresholded_prot")].copy()
    return None if sub.empty else sub.sort_values(["spearman", "r2", "n_test"], ascending=[False, False, False]).iloc[0].to_dict()

tree_detail_case = get_best_detail_row_for_case(bakeoff_df, tree_case, ALL_SEEDS[0]) if tree_case else None
mlp_detail_case = get_best_detail_row_for_case(bakeoff_df, mlp_case, ALL_SEEDS[0]) if mlp_case else None

def build_case_dataset(case_row):
    arm, threshold, feature_set, fold_i, drug, seed = str(case_row["arm"]), float(case_row["prot_max_missing_frac"]), str(case_row["feature_set"]), int(case_row["fold"]), str(case_row["compound_id"]), int(case_row["seed"])
    prot_df = prot_arm_data[arm].copy(); prot_df.index = prot_df.index.astype(str).str.strip(); prot_core = prot_df.reindex(core_cells)
    eligible_cells = sorted(prot_core.notna().any(axis=1)[lambda s: s].index.tolist())
    splits, _ = safe_group_splits(eligible_cells, groups_all.reindex(eligible_cells).fillna("Unknown").astype(str), N_SPLITS_DESIRED, seed)
    train_idx, test_idx = splits[fold_i]
    train_cells = [eligible_cells[int(i)] for i in train_idx]; test_cells = [eligible_cells[int(i)] for i in test_idx]
    force_indicators = (arm == TRACK2_ARM)
    feat_keys = parse_feature_set(feature_set)
    X_train_parts, X_all_parts, feature_names, modality_slices = [], [], [], {}
    start_col = 0
    if "rna" in feat_keys:
        Xtr, Xall, names = build_original_filtered_block(rna, train_cells, eligible_cells, "rna", "median", True)
        X_train_parts.append(Xtr); X_all_parts.append(Xall); feature_names += names; modality_slices["rna"] = slice(start_col, start_col + Xall.shape[1]); start_col += Xall.shape[1]
    if "cnv" in feat_keys:
        Xtr, Xall, names = build_original_filtered_block(cnv, train_cells, eligible_cells, "cnv", "median", True)
        X_train_parts.append(Xtr); X_all_parts.append(Xall); feature_names += names; modality_slices["cnv"] = slice(start_col, start_col + Xall.shape[1]); start_col += Xall.shape[1]
    if "mut" in feat_keys:
        Xtr, Xall, names = build_original_filtered_block(mut, train_cells, eligible_cells, "mut", "most_frequent", False)
        X_train_parts.append(Xtr); X_all_parts.append(Xall); feature_names += names; modality_slices["mut"] = slice(start_col, start_col + Xall.shape[1]); start_col += Xall.shape[1]
    if "prot" in feat_keys:
        Xtr, Xall, names = build_original_filtered_block(prot_core.loc[eligible_cells], train_cells, eligible_cells, "prot", "median", True, threshold, False, force_indicators)
        X_train_parts.append(Xtr); X_all_parts.append(Xall); feature_names += names; modality_slices["prot"] = slice(start_col, start_col + Xall.shape[1]); start_col += Xall.shape[1]
    X_all_full = np.concatenate(X_all_parts, axis=1) if X_all_parts else np.zeros((len(eligible_cells), 0), dtype=np.float32)
    pairs = drug_to_pairs[drug].copy(); pairs = pairs[pairs["depmap_id"].isin(eligible_cells)][["depmap_id", "y"]].groupby("depmap_id", as_index=False)["y"].mean()
    y_by_cell = dict(zip(pairs["depmap_id"].astype(str), pairs["y"].astype(float)))
    keep_train, keep_test = [c for c in train_cells if c in y_by_cell], [c for c in test_cells if c in y_by_cell]
    c2r = {c: i for i, c in enumerate(eligible_cells)}
    train_rows, test_rows = [c2r[c] for c in keep_train], [c2r[c] for c in keep_test]
    return {"Xtr": X_all_full[train_rows], "Xte": X_all_full[test_rows], "ytr": np.array([y_by_cell[c] for c in keep_train], dtype=float), "yte": np.array([y_by_cell[c] for c in keep_test], dtype=float), "feature_names": feature_names, "modality_slices": modality_slices, "train_cells": keep_train, "test_cells": keep_test}

# SHAP
if tree_detail_case is not None and HAS_SHAP:
    case = build_case_dataset(tree_detail_case); model_name = str(tree_detail_case["model"]).lower()
    mdl = make_model(model_name, int(tree_detail_case["seed"]), case["Xtr"].shape[1])
    if mdl is not None:
        mdl.fit(case["Xtr"], case["ytr"]); pred = mdl.predict(case["Xte"])
        print("Tree case", model_name, "Spearman", spearman_corr(case["yte"], pred), "R2", r2_score(case["yte"], pred))
        try:
            shap_values = shap.TreeExplainer(mdl).shap_values(case["Xte"]) if model_name in {"extratrees", "xgb"} else shap.Explainer(mdl.predict, pd.DataFrame(case["Xtr"], columns=case["feature_names"]).sample(min(200, len(case["Xtr"])), random_state=int(tree_detail_case["seed"])))(pd.DataFrame(case["Xte"], columns=case["feature_names"])).values
            shap_df = mean_abs_shap_df(shap_values, case["feature_names"])
            shap_df.to_csv(INTERP_DIR / "tree_case__shap_feature_rank.csv", index=False)
            top25 = shap_df.head(25).copy()
            plt.figure(figsize=(10.8, max(5, int(0.38 * len(top25) + 1))))
            plt.barh(range(len(top25)), top25.iloc[::-1]["mean_abs_shap"])
            plt.yticks(range(len(top25)), [wrap_label(x, 48) for x in top25.iloc[::-1]["feature"]])
            plt.title("Top features by mean |SHAP|"); plt.xlabel("Mean |SHAP|")
            finish_plot(INTERP_DIR / "tree_case__top25_mean_abs_shap.png")
            try_run_pathway_enrichment(shap_df["feature"].head(100).tolist(), INTERP_DIR / "tree_case__pathway_enrichment")
        except Exception as e:
            print("[SHAP] failed:", e)

# IG
if mlp_detail_case is not None and HAS_CAPTUM:
    case = build_case_dataset(mlp_detail_case)
    class AttributionMLP(nn.Module):
        def __init__(self, in_dim):
            super().__init__()
            self.net = nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), nn.Dropout(0.10), nn.Linear(256, 64), nn.ReLU(), nn.Linear(64, 1))
        def forward(self, x): return self.net(x).squeeze(-1)
    set_seeds(int(mlp_detail_case["seed"]))
    ig_model = AttributionMLP(case["Xtr"].shape[1]).to(TORCH_DEVICE)
    opt = torch.optim.Adam(ig_model.parameters(), lr=1e-3, weight_decay=1e-5); loss_fn = nn.MSELoss()
    xb, yb = torch.from_numpy(case["Xtr"].astype(np.float32)).to(TORCH_DEVICE), torch.from_numpy(case["ytr"].astype(np.float32)).to(TORCH_DEVICE)
    best_state, best_loss, bad_epochs = copy.deepcopy(ig_model.state_dict()), math.inf, 0
    for _ in range(80):
        ig_model.train(); loss = loss_fn(ig_model(xb), yb); opt.zero_grad(); loss.backward(); opt.step()
        current = float(loss.item())
        if current < (best_loss - 1e-6):
            best_loss, best_state, bad_epochs = current, copy.deepcopy(ig_model.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= 10: break
    ig_model.load_state_dict(best_state); ig_model.eval()
    with torch.no_grad():
        pred = ig_model(torch.from_numpy(case["Xte"].astype(np.float32)).to(TORCH_DEVICE)).detach().cpu().numpy()
    print("MLP case Spearman", spearman_corr(case["yte"], pred), "R2", r2_score(case["yte"], pred))
    try:
        ig = IntegratedGradients(ig_model)
        x_test_t = torch.from_numpy(case["Xte"].astype(np.float32)).to(TORCH_DEVICE)
        baseline = torch.zeros((1, case["Xte"].shape[1]), dtype=torch.float32, device=TORCH_DEVICE)
        ig_attr = ig.attribute(x_test_t, baselines=baseline, n_steps=50)
        ig_rank = pd.DataFrame({"feature": case["feature_names"], "mean_abs_ig": np.abs(ig_attr.detach().cpu().numpy()).mean(axis=0)}).sort_values("mean_abs_ig", ascending=False)
        ig_rank.to_csv(INTERP_DIR / "mlp_case__ig_feature_rank.csv", index=False)
        try_run_pathway_enrichment(ig_rank["feature"].head(100).tolist(), INTERP_DIR / "mlp_case__pathway_enrichment")
    except Exception as e:
        print("[IG] failed:", e)

# Attention surrogate
RUN_ATTENTION_SURROGATE = True
class ModalityAttentionRegressor(nn.Module):
    def __init__(self, dims_by_modality, hidden_dim=64, dropout=0.10):
        super().__init__(); self.modalities = [m for m, d in dims_by_modality.items() if d > 0]; self.towers = nn.ModuleDict(); self.attn_scores = nn.ModuleDict()
        for m, d in dims_by_modality.items():
            if d <= 0: continue
            self.towers[m] = nn.Sequential(nn.Linear(d, hidden_dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_dim, hidden_dim), nn.ReLU())
            self.attn_scores[m] = nn.Linear(hidden_dim, 1)
        self.head = nn.Sequential(nn.Linear(hidden_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 1))
    def forward(self, x_parts, return_attn=False):
        reps, logits = [], []
        for m in self.modalities:
            h = self.towers[m](x_parts[m]); reps.append(h.unsqueeze(1)); logits.append(self.attn_scores[m](h))
        reps, logits = torch.cat(reps, dim=1), torch.cat(logits, dim=1)
        attn = torch.softmax(logits, dim=1); fused = (attn * reps).sum(dim=1); out = self.head(fused).squeeze(-1)
        return (out, attn.squeeze(-1)) if return_attn else out

if RUN_ATTENTION_SURROGATE and tree_detail_case is not None:
    attn_case = build_case_dataset(tree_detail_case)
    dims_by_modality = {m: attn_case["modality_slices"][m].stop - attn_case["modality_slices"][m].start for m in attn_case["modality_slices"]}
    def split_by_modality(X, modality_slices): return {m: X[:, sl].astype(np.float32) for m, sl in modality_slices.items()}
    Xtr_parts_np, Xte_parts_np = split_by_modality(attn_case["Xtr"], attn_case["modality_slices"]), split_by_modality(attn_case["Xte"], attn_case["modality_slices"])
    model_attn = ModalityAttentionRegressor(dims_by_modality=dims_by_modality).to(TORCH_DEVICE)
    opt = torch.optim.Adam(model_attn.parameters(), lr=1e-3, weight_decay=1e-5); loss_fn = nn.MSELoss()
    ytr_t = torch.from_numpy(attn_case["ytr"].astype(np.float32)).to(TORCH_DEVICE)
    best_state, best_loss, bad_epochs = copy.deepcopy(model_attn.state_dict()), math.inf, 0
    for _ in range(80):
        model_attn.train(); x_parts_t = {m: torch.from_numpy(v).to(TORCH_DEVICE) for m, v in Xtr_parts_np.items()}
        loss = loss_fn(model_attn(x_parts_t), ytr_t); opt.zero_grad(); loss.backward(); opt.step()
        current = float(loss.item())
        if current < (best_loss - 1e-6):
            best_loss, best_state, bad_epochs = current, copy.deepcopy(model_attn.state_dict()), 0
        else:
            bad_epochs += 1
            if bad_epochs >= 10: break
    model_attn.load_state_dict(best_state); model_attn.eval()
    with torch.no_grad():
        x_parts_te = {m: torch.from_numpy(v).to(TORCH_DEVICE) for m, v in Xte_parts_np.items()}
        pred_te, attn_te = model_attn(x_parts_te, return_attn=True)
        pred_te, attn_te = pred_te.detach().cpu().numpy(), attn_te.detach().cpu().numpy()
    print("Attention surrogate Spearman", spearman_corr(attn_case["yte"], pred_te), "R2", r2_score(attn_case["yte"], pred_te))
    attn_df = pd.DataFrame(attn_te, columns=model_attn.modalities, index=attn_case["test_cells"]).reset_index().rename(columns={"index": "depmap_id"})
    attn_df.to_csv(INTERP_DIR / "attention_surrogate__per_sample_weights.csv", index=False)
    mean_attn = attn_df[model_attn.modalities].mean(axis=0).rename("mean_attention").reset_index().rename(columns={"index": "modality"})
    mean_attn.to_csv(INTERP_DIR / "attention_surrogate__mean_weights.csv", index=False)

# Optional GNN pilot
RUN_GNN_PILOT = False
if RUN_GNN_PILOT and HAS_PYG:
    print("Enable and adapt the optional GNN pilot block here if a STRING edge file is available.")
else:
    print("GNN pilot skipped. Set RUN_GNN_PILOT = True and ensure PyG + STRING edges are available.")

In [ ]:

artefact_index = {
    "detail_csv": str(merged_path),
    "summary_csv": str(summary_path),
    "best_threshold_per_config": str(OUT_REPORTS / "best_threshold_per_config.csv"),
    "best_threshold_per_arm": str(OUT_REPORTS / "best_threshold_per_arm.csv"),
    "threshold_lock_json": str(threshold_choice_path),
    "missingness_report_json": str(OUT_REPORTS / "missingness_report.json"),
    "figures_dir": str(FIG_DIR),
    "interpretability_dir": str(INTERP_DIR),
}
write_json(artefact_index, OUT_REPORTS / "artefact_index.json")
display(pd.DataFrame({"artefact": list(artefact_index.keys()), "path": list(artefact_index.values())}))
print("Artefact index written.")